In [20]:
from analysis.processors.base import BaseProcessor
from coffea.nanoevents import NanoAODSchema
from coffea import processor
import uproot
import warnings
NanoAODSchema.warn_missing_crossrefs = False
!voms-proxy-init --voms cms > /dev/null 2>&1

workflow = "btag_eff_ele"
year = "2022preEE"

fileset = {'2022preEE': {
     'TTTo2L2Nu_1': {
         'files': {'root://cms-xrd-global.cern.ch:1094//store/mc/Run3Summer22NanoAODv12/TTto2L2Nu_TuneCP5_13p6TeV_powheg-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_v5-v2/50000/d51aa7d0-59ab-4ce7-8852-3f9a1ec14bc6.root': 'Events'},
         'metadata': {'short_name': 'TTTo2L2Nu'}},
    'SingleMuonC_1': {
       'files': {"root://cms-xrd-global.cern.ch:1094//store/data/Run2022C/DoubleMuon/NANOAOD/22Sep2023-v1/50000/9f317280-5380-49f7-8a30-3e3bcd48dc9c.root": "Events"},
       'metadata': {'short_name': 'SingleMuon'}},
    'WJetsToLNu_120_HT1500to2500_1': {
        'files': {"root://cms-xrd-global.cern.ch//store/mc/Run3Summer22EENanoAODv12/WtoLNu-4Jets_MLNu-0to120_HT-100to400_TuneCP5_13p6TeV_madgraphMLM-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_postEE_v6-v3/50000/27e1554b-3d69-4277-8549-88a697014d63.root": "Events"},
        'metadata': {'short_name': 'WJetsToLNu'}
    }
}
          }

In [5]:
futures_run = processor.Runner(
    executor=processor.FuturesExecutor(workers=4, compression=None),
    schema=NanoAODSchema,
    savemetrics=False
)
out = futures_run(fileset[year], treename="Events", processor_instance=BaseProcessor(workflow=workflow, year=year, mode="virtual"))
out["metadata"]

Output()

Output()

There are 20773 events with muon pt outside of [26,200] GeV. Setting those entries to their initial value.
There are 20776 events with muon pt outside of [26,200] GeV. Setting those entries to their initial value.
There are 115 nan entries in the corrected pt. This might be due to the number of tracker layers hitting boundaries. Setting those entries to their initial value.


{'sumw': np.float32(3.364027e+06),
 'base': {'cutflow': {'initial': np.float32(3.364027e+06),
   'goodvertex': np.float32(3.364027e+06),
   'lumi': np.float32(3.364027e+06),
   'trigger': np.float32(1.0578381e+06),
   'trigger_match': np.float32(1.0471324e+06),
   'metfilters': np.float32(1.04664575e+06),
   'hemcleaning': np.float32(1.04664575e+06),
   'met_50': np.float32(742992.75),
   'tau_veto': np.float32(742992.75),
   'muon_veto': np.float32(599763.25),
   'exactly_one_electron': np.float32(282971.47)},
  'weighted_final_nevents': np.float64(255709.74085459043)}}

In [12]:
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
filename = list(fileset[year]['WJetsToLNu_120_HT1500to2500_1']['files'].keys())[0]
events = NanoEventsFactory.from_root(
    filename,
    treepath="Events",
    entry_stop=1_000,
    metadata={"dataset": "WJetsToLNu_120_HT1500to2500_1"},
    schemaclass=NanoAODSchema,
    mode="virtual",
).events()

### Why HTCondor dask cluster doesn't work in SWAN?

In [ ]:
from dask.distributed import Client
client = Client("tls://10.100.197.122:30795")
dask_run = processor.Runner( #this is new runner function for coffea 2025.10
    executor=processor.DaskExecutor(client=client, compression=None), #execute via dask workers
    schema=NanoAODSchema,
    chunksize=100_000,
    skipbadfiles=False,
    savemetrics=False,
)
#histograms = dask_run(test_fileset, processor_instance=MuonProcessor(workflow_path, year))
histograms = dask_run(fileset[year], treename="Events", processor_instance=BaseProcessor(workflow=workflow, year=year, mode="virtual"))

In [14]:
from coffea.dataset_tools import apply_to_fileset, max_chunks, max_files, preprocess
preprocessed_available, preprocessed_total = preprocess(
    fileset[year],
    step_size=100_000,
    align_clusters=False,
    skip_bad_files=False,
    recalculate_steps=False,
    files_per_batch=1,
    file_exceptions=(OSError,),
    save_form=False,
    uproot_options={},
    step_size_safety_factor=0.5,
)

In [ ]:
test_preprocessed_files = max_files(preprocessed_available, 1)
test_preprocessed = max_chunks(test_preprocessed_files, 3)
small_tg, small_rep = apply_to_fileset(
    data_manipulation=BaseProcessor(workflow=workflow, year=year, mode="dask"),
    fileset=test_preprocessed,
    schemaclass=NanoAODSchema,
    uproot_options={"allow_read_errors_with_report": (OSError, ValueError)},
)